Cell 1: Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import pandas as pd
from src import config
from src.evaluation import (
    build_test_set, paraphrase_claims, gold_rank, retrieval_metrics, run_rag, run_no_rag
)

pd.set_option("display.max_colwidth", 120)
config.EVAL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(config.FCTR_CLEAN_PATH)

/Users/simgecevik/Desktop/WT/teyit-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 20:33:54.314000 49721 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Cell 2: Build the test set and paraphrase it (only once)

In [2]:
test_path = config.EVAL_DIR / "test_set.parquet"

if test_path.exists():
    test = pd.read_parquet(test_path)
    print("Loaded existing test set")
else:
    test = build_test_set(df, n_per_class=50)
    test["query"] = paraphrase_claims(test["claim"].tolist())
    test.to_parquet(test_path, index=False)
    print("Created and saved a new test set")

print(test["binary_label"].value_counts())

Loaded existing test set
binary_label
FALSE    50
TRUE     50
Name: count, dtype: int64


Cell 2b: Regenerate paraphrases with the improved prompt (run once)

In [ ]:
#test["query"] = paraphrase_claims(test["claim"].tolist())
#test.to_parquet(test_path, index=False)
#print(test.loc[82, "query"])

6 Şubat depremlerinde Hatay’ın Erzin ilçesinde tek bir bina bile yıkılmadı.


Cell 3: Inspect paraphrases by hand

In [4]:
test[["claim", "query", "binary_label"]].sample(12, random_state=1)

,claim,query,binary_label
80,Bir Instagram sayfası tarafından 5 Haziran 2021 tarihinde yapılan paylaşımda dünyanın en sinirli ülkelerine yönelik ...,"Türkiye, 5 Haziran 2021 tarihinde yayımlanan dünyanın en sinirli ülkeleri listesinde 6. sırada yer alıyor.",TRUE
84,Bakan Pakdemirli'nin patates firmasında danışmanlık yaptığı iddiası,Bakan Pakdemirli bir patates firmasında danışmanlık yapıyor.,TRUE
33,‘Aynasız’ kelimesinin aynasız olarak üretilen sivil polis araçlarından geldiği iddiası,Aynasız sözcüğü üzerinde ayna barındırmayan sivil polis otomobillerinden türetilmiştir,FALSE
81,Singapur’da tren camlarının binalara gelince matlaştığı iddiası,Singapur’daki trenlerin camları binaların yakınına gelince matlaşıyor.,TRUE
93,Kızılay’ın Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttüğü iddiası,"Kızılay, Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttü.",TRUE
17,"İş adamı, mirasını doğduğu köyün sakinlerine bıraktı mı?",Ünlü bir iş adamı bütün servetini ve mirasını doğduğu köyün sakinlerine bıraktı.,FALSE
36,FDA’nın üçüncü dozu ölümlere neden olduğu gerekçesiyle onaylamadığı iddiası,FDA koronavirüs aşısının üçüncü dozunu insan ölümlerine yol açtığı için onaylamadı,FALSE
82,6 Şubat depremlerinde Hatay’ın Erzin ilçesinde hiçbir binanın yıkılmadığı iddiası,6 Şubat depremlerinde Hatay’ın Erzin ilçesinde tek bir bina bile yıkılmadı.,TRUE
69,Soruların bekçi alımı için yapılan sınavda sorulduğu iddiası,Bahsi geçen sorular bekçi alımı için yapılan sınavda soruldu.,TRUE
65,THY'ye ait insani yardım uçaklarının dönüşte vatandaşları taşıdığı iddiası,THY'ye ait insani yardım uçakları dönüş yolunda vatandaşları da taşıdı.,TRUE


In [5]:
# Cell 3b: rows that had problems with the first prompt
test.loc[[33, 36, 39, 69, 82, 93], ["claim", "query"]]

,claim,query
33,‘Aynasız’ kelimesinin aynasız olarak üretilen sivil polis araçlarından geldiği iddiası,Aynasız sözcüğü üzerinde ayna barındırmayan sivil polis otomobillerinden türetilmiştir
36,FDA’nın üçüncü dozu ölümlere neden olduğu gerekçesiyle onaylamadığı iddiası,FDA koronavirüs aşısının üçüncü dozunu insan ölümlerine yol açtığı için onaylamadı
39,@gulderenmp adlı hesabın Memleket Partisi’den Gülderen Zeğerek Özafacan’a ait olduğu iddiası,Twitter platformundaki @gulderenmp hesabı Memleket Partisi üyelerinden Gülderen Zeğerek Özafacan'a aittir
69,Soruların bekçi alımı için yapılan sınavda sorulduğu iddiası,Bahsi geçen sorular bekçi alımı için yapılan sınavda soruldu.
82,6 Şubat depremlerinde Hatay’ın Erzin ilçesinde hiçbir binanın yıkılmadığı iddiası,6 Şubat depremlerinde Hatay’ın Erzin ilçesinde tek bir bina bile yıkılmadı.
93,Kızılay’ın Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttüğü iddiası,"Kızılay, Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttü."


In [6]:
# Cell 3c: a fresh sample, to check that the fix generalizes
test[["claim", "query", "binary_label"]].sample(15, random_state=7)

,claim,query,binary_label
37,Sezai Temelli'nin İsmet İnönü'nün akrabası olduğu iddiası,Sezai Temelli ile İsmet İnönü arasında akrabalık bağı bulunuyor,FALSE
26,Bilim insanlarının zamanı geri almayı başardığı iddiası,Bilim insanları yapılan çalışmalar sonucunda zamanı geriye sardı,FALSE
78,Bir Twitter sayfası tarafından 20 Mayıs 2021 tarihinde yapılan paylaşıma göre Türkiye Uzay Ajansı’nın 1 milyon 62 bi...,Türkiye Uzay Ajansı 1 milyon 62 bin 919 Türk lirasına biri VIP minibüs olmak üzere 5 araç ve iki şoför hizmetini iki...,TRUE
91,"Bir Instagram hesabı tarafından 1 Ekim 2022 tarihinde yapılan paylaşımda, Sayıştay raporuna göre Cumhurbaşkanlığı Kü...",Sayıştay raporuna göre Cumhurbaşkanlığı Külliyesi’nin günlük harcamaları 11 milyon lira.,TRUE
49,Covid-19 sürüntü testinin beyni koruyan bariyere zarar verdiği iddiası,Covid-19 sürüntü testi beyni koruyan bariyere zarar veriyormuş.,FALSE
15,Bir Twitter hesabı tarafından 15 Nisan 2023 tarihinde yapılan paylaşımda IMF tarafından dünya genelinde tek geçerli ...,"IMF, dünya genelinde tek geçerli para birimi olacak merkez bankası dijital parası olarak Unicoin’i piyasaya çıkardı.",FALSE
93,Kızılay’ın Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttüğü iddiası,"Kızılay, Başkent Gaz’ın yaptığı bağışla Ensar Vakfı’yla proje yürüttü.",TRUE
71,Hollanda’da bir kilise çanlarıyla 'Uzun İnce Bir Yoldayım' çalındığı iddiası,Hollanda’da bir kilise çanlarıyla 'Uzun İnce Bir Yoldayım' çalındı.,TRUE
86,Genç Müslüman Kadınlar İçin Liderlik programına 1 milyon 163 bin TL harcandığı iddiası,Genç Müslüman Kadınlar İçin Liderlik programı için 1 milyon 163 bin TL harcandı.,TRUE
22,Dolmalık biberin bir cinsiyeti olduğu iddiası,Dolmalık biberlerin belirli bir cinsiyeti bulunuyor,FALSE


In [7]:
import re

def word_set(text):
    """Set of lowercase words in a text."""
    return set(re.findall(r"\w+", text.lower()))

def word_overlap(a, b):
    """Jaccard similarity: shared words divided by all distinct words."""
    words_a, words_b = word_set(a), word_set(b)
    return len(words_a & words_b) / len(words_a | words_b)

test["word_overlap"] = [word_overlap(c, q) for c, q in zip(test["claim"], test["query"])]

print(test["word_overlap"].describe().round(2))
print("\nNear copies (overlap > 0.7):", (test["word_overlap"] > 0.7).sum())

count    100.00
mean       0.48
std        0.18
min        0.05
25%        0.33
50%        0.46
75%        0.63
max        0.88
Name: word_overlap, dtype: float64

Near copies (overlap > 0.7): 12


In [8]:
test.sort_values("word_overlap")[["claim", "query", "word_overlap"]].head(5)

,claim,query,word_overlap
20,Fransa’daki polislerin yurttaşlara yaptırım uygulamama kararı alıp görevini bıraktığı iddiası,Fransa içindeki polisler vatandaşlara ceza kesmeyi reddederek görevlerinden ayrıldılar,0.052632
31,Bir Instagram sayfasında 18 Mart 2021 tarihinde yapılan bir paylaşımda Tanzanya Devlet Başkanı’nın Covid-19 hakkında...,Tanzanya Devlet Başkanı koronavirüsle ilgili yaptığı açıklamalar yüzünden hayatına son verildi,0.095238
36,FDA’nın üçüncü dozu ölümlere neden olduğu gerekçesiyle onaylamadığı iddiası,FDA koronavirüs aşısının üçüncü dozunu insan ölümlerine yol açtığı için onaylamadı,0.105263
81,Singapur’da tren camlarının binalara gelince matlaştığı iddiası,Singapur’daki trenlerin camları binaların yakınına gelince matlaşıyor.,0.142857
30,Bir Twitter kullanıcısı tarafından 14 Aralık 2021 tarihinde yapılan paylaşımda Sağlık Bakanlığı’nın Covid-19 öncesin...,Sağlık Bakanlığı koronavirüs salgını başlamadan önce logosunu satanizm sembolüyle değiştirdi,0.142857


In [9]:
for i in [31, 30]:
    print(test.loc[i, "claim"])
    print("→", test.loc[i, "query"])
    print()

Bir Instagram sayfasında 18 Mart 2021 tarihinde yapılan bir paylaşımda Tanzanya Devlet Başkanı’nın Covid-19 hakkında yaptığı konuşma nedeniyle öldürüldüğü iddia edildi. Paylaşım 8 binden fazla beğeni almış ve 12 binden fazla paylaşılmış. Paylaşımda geçen meten şu şekilde:
→ Tanzanya Devlet Başkanı koronavirüsle ilgili yaptığı açıklamalar yüzünden hayatına son verildi

Bir Twitter kullanıcısı tarafından 14 Aralık 2021 tarihinde yapılan paylaşımda Sağlık Bakanlığı’nın Covid-19 öncesinde logosunu satanizm figürüyle değiştirdiği iddia edildi.
→ Sağlık Bakanlığı koronavirüs salgını başlamadan önce logosunu satanizm sembolüyle değiştirdi



### Paraphrase quality check

- **Prompt v1:** about half of 12 sampled paraphrases had problems:
  3 changed the meaning ("no building collapsed" → "no building was damaged"),
  3 dropped key entities ("FDA" → "a health organisation", a username removed).
- **Cause:** the prompt pushed for different wording without requiring entities
  or the exact meaning to be kept.
- **Prompt v2:** added rules to keep names, numbers and the exact meaning, with an example.
  All 6 problem rows fixed; no meaning changes in 15 new samples.
- **Trade-off:** v2 is conservative; some queries are near copies of the archive claim
  (word overlap > 0.7: 12 of 100). Scenario A results may therefore be slightly optimistic.
- Lowest-overlap queries were also checked by hand; meaning was preserved in all of them.

Cell 4: Retrieval metrics (no Gemini calls)

In [10]:
ranks_query = [gold_rank(q, d) for q, d in zip(test["query"], test["doc_id"])]
ranks_claim = [gold_rank(c, d) for c, d in zip(test["claim"], test["doc_id"])]

retrieval_results = pd.DataFrame({
    "paraphrased query": retrieval_metrics(ranks_query),
    "original claim text": retrieval_metrics(ranks_claim),
}).round(3)
retrieval_results

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7291.00it/s]


,paraphrased query,original claim text
recall@1,0.990,0.990
recall@3,1.000,1.000
recall@5,1.000,1.000
recall@10,1.000,1.000
mrr,0.995,0.995


In [16]:
# Cell 4b: sanity checks for retrieval
from src.retrieval import retrieve

print("Wrong id      :", gold_rank(test.loc[0, "query"], "fctr-999999"))
print("Unrelated text:", gold_rank("Bugün hava çok güzel, parka gittim", test.loc[0, "doc_id"]))

for q, d, r in zip(test["query"], test["doc_id"], ranks_query):
    if r != 1:
        top = retrieve(q, top_k=1)[0]
        print(f"\nrank {r} | {d}")
        print("  query  :", q)
        print("  archive:", test.loc[test["doc_id"] == d, "claim"].iloc[0])
        print("  ranked first instead:", top["doc_id"], "|", top["claim"][:100])

Wrong id      : None
Unrelated text: None

rank 2 | fctr-5366
  query  : Tanzanya Devlet Başkanı koronavirüsle ilgili yaptığı açıklamalar yüzünden hayatına son verildi
  archive: Bir Instagram sayfasında 18 Mart 2021 tarihinde yapılan bir paylaşımda Tanzanya Devlet Başkanı’nın Covid-19 hakkında yaptığı konuşma nedeniyle öldürüldüğü iddia edildi. Paylaşım 8 binden fazla beğeni almış ve 12 binden fazla paylaşılmış. Paylaşımda geçen meten şu şekilde:
  ranked first instead: fctr-5498 | Bir Instagram kullanıcısı tarafından 24 Aralık 2020 tarihinde yapılan paylaşımda Tanzanya’da yaşadığ


### Retrieval findings

- Recall@1 = 0.99, Recall@3 = 1.00, MRR = 0.995, identical for paraphrased and original queries.
- Sanity checks passed: a non-existent id and an unrelated query are not ranked first.
- Why so high: every chunk carries the claim as a context header, paraphrases keep
  distinctive entities (names, places), and the archive is small and diverse.
- Limitation: test queries contain the key entities of each claim. Vague queries
  or image/video-based claims would likely be harder.
- Consequence: errors in Scenario A will come almost entirely from the generation step.
- The only query not ranked first (fctr-5366, Tanzania) was ranked second, behind another
  fact-check about Tanzania and Covid-19; its archive claim text is also truncated.

Cell 5: Scenario A, known claim (correct article is in the archive)

In [11]:
smoke = run_rag(test.head(3), config.EVAL_DIR / "rag_known.jsonl")
smoke[["doc_id", "expected", "raw_verdict", "verdict", "guardrail_applied"]]

0 cached, 3 to run


,doc_id,expected,raw_verdict,verdict,guardrail_applied
0,fctr-2664,FALSE,FALSE,FALSE,False
1,fctr-4391,FALSE,FALSE,FALSE,False
2,fctr-5044,FALSE,FALSE,FALSE,False


Cell 5b: Run Rag

In [ ]:
rag_known = run_rag(test, config.EVAL_DIR / "rag_known.jsonl")
print("Accuracy:", round((rag_known["verdict"] == rag_known["expected"]).mean(), 3))

97 cached, 3 to run
Accurancy: 0.97


Cell 6: Scenario B, new claim (own article excluded from the archive)

In [14]:
rag_new = run_rag(test, config.EVAL_DIR / "rag_new.jsonl", exclude_own=True)
print("Accuracy:", round((rag_new["verdict"] == rag_new["expected"]).mean(), 3))

0 cached, 100 to run
 10 / 100 done
 20 / 100 done
 30 / 100 done
Gemini returned 503, retrying in 5s.
 40 / 100 done
 50 / 100 done
 60 / 100 done
 70 / 100 done
 80 / 100 done
 90 / 100 done
 100 / 100 done
Accuracy: 0.83


Cell 7: Baseline without RAG (Gemini's own knowledge only)

In [15]:
no_rag = run_no_rag(test, config.EVAL_DIR / "no_rag.jsonl")
print("Accuracy:", round((no_rag["verdict"] == no_rag["expected"]).mean(), 3))
print(no_rag["verdict"].value_counts())

0 cached, 100 to run
 10 / 100 done
 20 / 100 done
Gemini returned 503, retrying in 5s.
 30 / 100 done
 40 / 100 done
 50 / 100 done
 60 / 100 done
 70 / 100 done
 80 / 100 done
 90 / 100 done
 100 / 100 done
Accuracy: 0.61
verdict
FALSE                  37
TRUE                   35
NOT_ENOUGH_EVIDENCE    28
Name: count, dtype: int64


In [17]:
# Cell 8: quick look at the results (reads the cache, no API calls)
print("Scenario A errors:")
errors_a = rag_known[rag_known["verdict"] != rag_known["expected"]]
print(errors_a[["doc_id", "query", "expected", "verdict"]].to_string(), "\n")

print("Scenario B verdicts:")
print(rag_new["verdict"].value_counts(), "\n")

print("Guardrail applied:")
print("  Scenario A:", rag_known["guardrail_applied"].sum())
print("  Scenario B:", rag_new["guardrail_applied"].sum())

Scenario A errors:
       doc_id                                                                                                               query expected              verdict
14   fctr-887                                            Cinsiyet uyum süreci ile ilgili merak edilen tüm gerçekler ortaya çıktı.    FALSE  NOT_ENOUGH_EVIDENCE
55  fctr-4631                                                          Yunanistan'da asgari ücretle 1088 yumurta alınabiliyormuş.     TRUE                MIXED
83  fctr-4026  2023 yılında Aydın Çıldır, Balıkesir Merkez, Çanakkale Gökçeada ve Uşak havalimanlarında hiç yolcu trafiği olmadı.     TRUE  NOT_ENOUGH_EVIDENCE 

Scenario B verdicts:
verdict
NOT_ENOUGH_EVIDENCE    83
FALSE                  13
TRUE                    4
Name: count, dtype: int64 

Guardrail applied:
  Scenario A: 0
  Scenario B: 0
